In [1]:
import os
import pandas as pd
import numpy as np
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
from sklearn.model_selection import train_test_split
import mlflow
import dagshub
from dotenv import load_dotenv
import joblib



In [2]:
DATA_PATH = "../data/disease_dataset.csv"  
df = pd.read_csv(DATA_PATH)

In [11]:
df.columns

Index(['lat', 'lon', 'observation_time', 'temp_c', 'humidity',
       'precipitation_mm', 'wind_speed', 'pressure', 'hour', 'day', 'month',
       'temp_rolling_mean', 'humidity_rolling_mean', 'precip_rolling_sum',
       'wind_rolling_mean', 'disease_risk'],
      dtype='object')

In [13]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 78912 entries, 0 to 78911
Data columns (total 16 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   lat                    78912 non-null  float64
 1   lon                    78912 non-null  float64
 2   observation_time       78912 non-null  object 
 3   temp_c                 78912 non-null  float64
 4   humidity               78912 non-null  float64
 5   precipitation_mm       78912 non-null  float64
 6   wind_speed             78912 non-null  float64
 7   pressure               78912 non-null  float64
 8   hour                   78912 non-null  int64  
 9   day                    78912 non-null  int64  
 10  month                  78912 non-null  int64  
 11  temp_rolling_mean      78912 non-null  float64
 12  humidity_rolling_mean  78912 non-null  float64
 13  precip_rolling_sum     78912 non-null  float64
 14  wind_rolling_mean      78912 non-null  float64
 15  di

In [ ]:
load_dotenv()
dagshub.init(repo_owner='IbrahimFaye', repo_name='weather-agri', mlflow=True)
mlflow.set_experiment("disease_risk_prediction")

DATA_PATH = "../data/disease_dataset.csv"  
df = pd.read_csv(DATA_PATH)

print(df["disease_risk"].value_counts(normalize=True))

def prepare_data(df):
    drop_cols = ["observation_time"]
    X = df.drop(columns=drop_cols + ["disease_risk"])
    y = df["disease_risk"]
    return X, y

X, y = prepare_data(df)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

def train_model():
    with mlflow.start_run(run_name="GBC") as run:
        model = GradientBoostingClassifier(
            n_estimators=200,
            learning_rate=0.05,
            max_depth=4,
            random_state=42
        )
        model.fit(X_train, y_train)
        preds = model.predict(X_test)

        acc = accuracy_score(y_test, preds)
        f1 = f1_score(y_test, preds)

        print(f"✅ Disease Model — Accuracy: {acc:.3f} | F1: {f1:.3f}")

        mlflow.log_metrics({"accuracy": acc, "f1_score": f1})
        mlflow.log_params({
            "n_estimators": 200,
            "learning_rate": 0.05,
            "max_depth": 4
        })

        importances = dict(zip(X.columns, model.feature_importances_))
        mlflow.log_dict(importances, "feature_importance.json")

        os.makedirs("artifacts", exist_ok=True)
        joblib.dump(model, "artifacts/disease_model.pkl", compress=6)
        mlflow.log_artifact("artifacts/disease_model.pkl", artifact_path="models")

        print("🔗 Résultats visibles sur DagsHub")

if __name__ == "__main__":
    train_model()


Initialized MLflow to track repo "IbrahimFaye/weather-agri"

Repository IbrahimFaye/weather-agri initialized!

disease_risk
0    0.920912
1    0.079088
Name: proportion, dtype: float64
